# Reproducing the Experiment

This notebook walks through all steps to reproduce the results from:
> **Harmonic Representations in Symbolic Music Transformers are Localized and Controllable**  
> Adi Silberschein, Megan Wei, Tamar Rott Shaham

By default all outputs are written to `reproduce_output/`.

**Prerequisites:** install dependencies with `pip install -e .`

> ⚙️ **Run the cell below first.** Every step uses paths relative to the repo root; since this
> notebook lives in `reproduce/`, the setup cell pins the working directory to the repo root so
> the rest of the notebook works no matter where the Jupyter kernel starts.

In [ ]:
# ⚙️ Run first: pin the working directory to the repo root (the dir with pyproject.toml + reproduce/).
# The notebook lives in reproduce/, so a freshly-opened kernel may start there; walk up until the
# repo root is found so every relative path below ('reproduce/…', 'config/…', 'reproduce_output/…') resolves.
import os
while not (os.path.isfile('pyproject.toml') and os.path.isdir('reproduce')):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        raise RuntimeError('Repo root not found — open this notebook from inside the cloned repo.')
    os.chdir(parent)
print('working directory:', os.getcwd())

In [ ]:
# 📄 Figure export — set SAVE_FIGURES = True to also write every figure below to PDF.
import os
import matplotlib.pyplot as plt

SAVE_FIGURES = False
FIGURE_DIR = 'reproduce_output/figures'

# --- Paper geometry ------------------------------------------------------------
# Every figure below is authored at the size it is *printed* at, so that
# \includegraphics[width=\columnwidth]{...} scales it 1:1 and a 6.5 pt label in the
# PDF is a 6.5 pt label on the page. Authoring at, say, 11 in wide and letting LaTeX
# shrink it into one column divides every font size by ~3.4 — which is exactly why the
# earlier exports came out unreadable beside the paper's own figures.
COL_W = 3.25     # \columnwidth of a two-column paper, in inches
TEXT_W = 6.75    # \textwidth — for the two wide figures that need a figure* environment
BASE_FS = 6.5    # tick-label / legend size in points, measured off the paper's Figure 2

plt.rcParams.update({
    'figure.dpi': 200,          # keeps the inline previews readable at these small sizes
    'savefig.dpi': 300,
    'pdf.fonttype': 42,         # embed TrueType, not Type 3 (arXiv-safe)
    'ps.fonttype': 42,
    'font.size': BASE_FS,
    'axes.titlesize': BASE_FS + 1,
    'axes.labelsize': BASE_FS + 1,
    'xtick.labelsize': BASE_FS,
    'ytick.labelsize': BASE_FS,
    'legend.fontsize': BASE_FS,
    'legend.frameon': False,    # paper legends sit bare inside the axes
    'legend.handlelength': 1.1,
    'legend.handletextpad': 0.5,
    'legend.labelspacing': 0.3,
    'legend.borderaxespad': 0.2,
    'axes.linewidth': 0.6,      # hairlines, so nothing looks heavy at print size
    'grid.linewidth': 0.5,
    'lines.linewidth': 1.2,
    'lines.markersize': 3.0,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.0, 'ytick.major.size': 2.0,
    'xtick.major.pad': 2.0, 'ytick.major.pad': 2.0,
})


def save_fig(fig, name):
    """Write `fig` to FIGURE_DIR/<name>.pdf (vector, at print size) when SAVE_FIGURES is on."""
    if not SAVE_FIGURES:
        return
    os.makedirs(FIGURE_DIR, exist_ok=True)
    path = os.path.join(FIGURE_DIR, f'{name}.pdf')
    # Keep pad_inches tiny: tight_layout has already reserved the margins, and the default
    # 0.1 in pad would inflate a 3.25 in figure to 3.45 in and quietly undo the 1:1 scale.
    fig.savefig(path, format='pdf', bbox_inches='tight', pad_inches=0.01)
    print(f'saved {path}')


---
## Step 0 — Download the model weights

Fetches the Aria checkpoint (`model-gen.safetensors`, ~2.6 GB) from the public HuggingFace repo
[`loubb/aria-medium-base`](https://huggingface.co/loubb/aria-medium-base) into
`config/models/aria-medium-base/` — where the pipeline expects it (`reproduce/consts.py`). Run
once on a fresh clone; it **skips the download** if the file is already present.

Network + disk only (no GPU). The small model-config JSONs (`config/models/medium-*.json`) are
already in the repo, so only the large weight file is downloaded.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/download_model.py'], check=True)

---
## Step 1 — Download JSB Chorales

Downloads the JSB Chorales dataset from [czhuang/JSB-Chorales-dataset](https://github.com/czhuang/JSB-Chorales-dataset)
and converts it to MIDI format (16th-note resolution).

Output: `reproduce_output/jsb_chorales_midi/{train,valid,test}_16th/`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/download_jsb_chorales.py'], check=True)

---
## Step 2 — Prepare Chorale Corpus

Runs harmonic analysis, detects cadence points, cuts MIDIs at cadences,
and creates minorized (counterfactual) versions — for both major and minor chorales.

Output:
- `reproduce_output/data/major_chorale_corpus/`
- `reproduce_output/data/minor_chorale_corpus/`

In [ ]:
import subprocess, sys

for mode in ['major', 'minor']:
    subprocess.run([
        sys.executable, 'reproduce/prepare_chorale_data.py', 'all',
        '--mode', mode,
        '--input', 'reproduce_output/jsb_chorales_midi/train_16th',
        '--output', f'reproduce_output/data/{mode}_chorale_corpus',
    ], check=True)

---
## Step 3 — Extract Activations for Patching

Runs each factual (original) cut and its minorized (counterfactual) pair through
Aria with activation hooks, for both the major and minor corpora. For every
(chorale, cut, seed) it saves:
- the **residual stream at the patch position** (the last prompt token), all layers
  — this is the patch source for the per-layer patching experiment;
- **logits/probs at the first 30 generated positions** — to measure the continuation.

No KV cache is saved. This produces the `original_activations/` and
`minorized_activations/` directories that the per-layer patching step consumes.

Output: `reproduce_output/patching/<timestamp>_reproduce_{major,minor}/`


> ⚠️ **Run this step on a GPU, not in the notebook.** From the repo root, in a GPU
> allocation with the aria env active:
>
> ```bash
> reproduce/extract_patching_activations.sh        # both corpora (or: major | minor)
> ```
>
> **Output:** `reproduce_output/patching/<timestamp>_reproduce_{major,minor}/`
> **Time (one A10):** major (89 chorales) ~50 min, minor (59) ~35 min, **both ~1.5 h.**

---
## Step 4 — Resulting harmony: original vs minorized (paper Figure 2)

For each generated continuation we label its **first chord after the cut** (the notes at the
first onset following the prompt) with a roman numeral relative to the chorale's key, then
compare the distribution for the **original** vs **minorized** prompts (no patching yet), for
the major-source and minor-source corpora.

Uses the most recent Step 3 run of each corpus. Runs on CPU — no GPU needed. The x-axis shows
the curated roman-numeral categories from the paper (`PAPER_CATEGORIES`).

In [ ]:
import os, sys, json, glob
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

sys.path.insert(0, os.path.abspath('reproduce'))
from ariautils.tokenizer import AbsTokenizer
from utils import resulting_harmony_rn, _parse_key_from_chords_txt

tokenizer = AbsTokenizer()

# Curated roman-numeral categories per source corpus (paper Figure 2).
PAPER_CATEGORIES = {
    'major': ['I', 'IV', 'VI', 'ii', 'vi', 'vi6', 'IV6', 'I6'],
    'minor': ['I', 'VI', 'i', 'I#3', 'VI6', 'i6', 'iv6', 'VI7'],
}

def resulting_harmony_distribution(corpus):
    """Label the first continuation chord of every (chorale, cut, seed) in the
    most recent Step 3 run for `corpus`. Returns (orig_counts, minor_counts, n)."""
    run = sorted(glob.glob(f'reproduce_output/patching/*_reproduce_{corpus}'))[-1]
    info = json.load(open(os.path.join(run, 'experiment_info.json')))
    data_dir, n_gen = info['data_dir'], info['capture_generated_tokens']

    key_cache = {}
    def get_key(chorale):
        if chorale not in key_cache:
            p = os.path.join(data_dir, f'{chorale}_chords.txt')
            key_cache[chorale] = _parse_key_from_chords_txt(p) if os.path.exists(p) else None
        return key_cache[chorale]

    orig, minor, n = Counter(), Counter(), 0
    for meta_path in glob.glob(os.path.join(run, 'chorale_*', 'cut_tick_*', 'seed_*',
                                            'original_activations', 'metadata.json')):
        seed_dir = os.path.dirname(os.path.dirname(meta_path))
        chorale = 'chorale_' + seed_dir.split('/chorale_')[1].split('/')[0]
        key = get_key(chorale)
        minor_path = os.path.join(seed_dir, 'minorized_activations', 'metadata.json')
        if key is None or not os.path.exists(minor_path):
            continue
        o = resulting_harmony_rn(json.load(open(meta_path)), key, tokenizer, n_gen)
        m = resulting_harmony_rn(json.load(open(minor_path)), key, tokenizer, n_gen)
        if o and o != '-':
            orig[o] += 1
        if m and m != '-':
            minor[m] += 1
        n += 1
    print(f'{corpus}: {os.path.basename(run)}  (n={n})')
    return orig, minor, n

# --- two-panel figure, sized and coloured to match the paper's Figure 2 ---
# Colours sampled from the paper's own bars (indigo/grass, ~viridis 0.20/0.77);
# the previous viridis(0.15)/viridis(0.62) pair read as violet/teal beside it.
c_orig, c_min = '#424483', '#7AC76E'
fig, axes = plt.subplots(1, 2, figsize=(COL_W, COL_W / 3.4), sharey=True)
for ax, (corpus, title) in zip(axes, [('major', 'Major-chorale source'),
                                      ('minor', 'Minor-chorale source')]):
    orig, minor, n = resulting_harmony_distribution(corpus)
    ot, mt = sum(orig.values()), sum(minor.values())
    cats = PAPER_CATEGORIES[corpus]
    x, w = np.arange(len(cats)), 0.4
    ax.bar(x - w/2, [100 * orig[c] / ot for c in cats], w, label='Original', color=c_orig)
    ax.bar(x + w/2, [100 * minor[c] / mt for c in cats], w, label='Minorized', color=c_min)
    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.set_title(title)
    ax.yaxis.set_major_locator(MultipleLocator(20))   # 0/20/../80, as in the paper
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('% of samples')
axes[1].legend(loc='upper right')   # bare frame + compact swatches come from rcParams
fig.tight_layout(pad=0.3)
save_fig(fig, 'figure2_resulting_harmony')
plt.show()

---
## Step 5 — Per-layer patching

For each (chorale, cut, seed) and each layer *i*, patch the residual stream at the last
prompt position with the saved **minorized** hidden state at layer *i*, then generate a
continuation. The prompt KV is recomputed once per seed via a prefill (no `kv_cache.pt`
from Step 3 needed).

> ⚠️ **Run this step on a GPU, not in the notebook.** From the repo root, in a GPU
> allocation with the aria env active:
>
> ```bash
> reproduce/run_per_layer_patching.sh              # both corpora
> # reproduce/run_per_layer_patching.sh major      # one corpus only
> ```
>
> **The plain command adapts to the GPUs it sees:**
> - **2+ GPUs** → auto-splits: **major → GPU 0, minor → GPU 1** in parallel (each logs to
>   `reproduce_output/patching/step5_{major,minor}.log`; `tail -f` to watch).
> - **1 GPU** → runs the two corpora **sequentially** on that GPU.
> - `SERIAL=1 reproduce/run_per_layer_patching.sh` forces one-GPU sequential even with 2 GPUs.
>
> **Output (new dirs only):** under each seed,
> `per_layer_last_position_patching/layer_<i>_patch/{output.mid, activations/}`
> **Time (A10):** ~32 s/seed → major (~550 seeds) ~5 h, minor (~270) ~2.5 h. One GPU ~7–8 h;
> split across 2 GPUs **~5 h**. Resumable — re-running skips already-finished patches.

---
## Step 6 — Causal effect by layer (paper Figure 3)

Interchange intervention accuracy (IIA) by layer. For each (chorale, cut, seed) we patch the
**minorized** hidden state at the last prompt position into the **original** run at layer *i*
and let generation continue.

**How success is scored — the *disambiguating note* criterion.** Take the resolution chord each
run produces (the notes at the first onset after the prompt). The original and the minorized
resolutions share most of their pitch classes; what tells them apart are the pitch classes
belonging to only one of the two. So we take the **first note of the patched run's resolution
chord** and ask which harmony its pitch class is compatible with:

| the patched note's pitch class is… | counted as |
|---|---|
| in the minorized chord only | **success** |
| in the original chord only | **failure** |
| in **both** (a shared tone) | ignored — not disambiguating |
| in **neither** | ignored |

$$\mathrm{IIA}(\ell)=\frac{n_{\text{minor only}}}{n_{\text{minor only}}+n_{\text{orig only}}}$$

Shaded regions are **95% Wilson score intervals** on that proportion (the plotted line is the
Wilson centre), matching the paper. Chance is 0.5.

Early layers (0–5) carry little causal harmonic information, so IIA sits near chance; the effect
emerges around layer 6 and grows through the deeper layers.

Runs on CPU. **Requires Step 5 to have finished** (reads `per_layer_last_position_patching/`).

In [ ]:
import os, sys, json, glob
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('reproduce'))
from ariautils.tokenizer import AbsTokenizer
from utils import resulting_harmony_notes

tokenizer = AbsTokenizer()
NUM_LAYERS = 16


def wilson_ci(successes, n, z=1.96):
    """Wilson score interval for a proportion: (centre, half-width)."""
    if n == 0:
        return np.nan, 0.0
    p = successes / n
    z2 = z * z
    denom = 1 + z2 / n
    centre = (p + z2 / (2 * n)) / denom
    spread = z * np.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / denom
    return centre, spread


def resolution_notes(meta_path):
    """Notes of the resolution chord produced by one run (empty list if none)."""
    return resulting_harmony_notes(json.load(open(meta_path)), tokenizer)


def classify_first_notes(corpus):
    """Per-layer disambiguating-note counts for the latest Step 5 run of `corpus`.

    For every (chorale, cut, seed) pair we take the pitch-class sets of the
    original and minorized resolution chords, then for each layer classify the
    first note of the patched run's resolution chord as belonging to the
    minorized chord only, the original chord only, both, or neither.
    Returns (minor_only, orig_only, n_pairs)."""
    run = sorted(glob.glob(f'reproduce_output/patching/*_reproduce_{corpus}'))[-1]

    minor_only = np.zeros(NUM_LAYERS, dtype=int)
    orig_only = np.zeros(NUM_LAYERS, dtype=int)
    n_pairs = 0
    for mp in glob.glob(os.path.join(run, 'chorale_*', 'cut_tick_*', 'seed_*',
                                     'minorized_activations', 'metadata.json')):
        seed_dir = os.path.dirname(os.path.dirname(mp))
        base = os.path.join(seed_dir, 'per_layer_last_position_patching')
        omp = os.path.join(seed_dir, 'original_activations', 'metadata.json')
        if not os.path.exists(omp) or not os.path.isdir(base):
            continue
        orig_pcs = {n['midi'] % 12 for n in resolution_notes(omp)}
        minor_pcs = {n['midi'] % 12 for n in resolution_notes(mp)}
        if not orig_pcs or not minor_pcs:
            continue
        n_pairs += 1
        for i in range(NUM_LAYERS):
            lp = os.path.join(base, f'layer_{i}_patch', 'activations', 'metadata.json')
            if not os.path.exists(lp):
                continue
            notes = resolution_notes(lp)
            if not notes:
                continue
            pc = notes[0]['midi'] % 12          # first note of the patched resolution
            in_orig, in_minor = pc in orig_pcs, pc in minor_pcs
            if in_minor and not in_orig:
                minor_only[i] += 1
            elif in_orig and not in_minor:
                orig_only[i] += 1
            # in both / in neither -> not disambiguating, ignored
    return minor_only, orig_only, n_pairs


# --- Figure 3: IIA per layer with 95% Wilson CI bands ---
cmap = plt.cm.viridis
styles = [('major', cmap(0.1), 'o', 'Major-chorale source'),
          ('minor', cmap(0.9), 's', 'Minor-chorale source')]

# Aspect measured off the paper's own Figure 3: it is placed in a fixed
# \columnwidth x 1.43 in box (\includegraphics with both width and height set), so a
# figure authored at 3.25 x 2.02 in gets its height squashed ~28% to fit -- which reads
# as the whole plot being stretched sideways next to the paper's own version. Authoring
# at that box's 2.26 aspect means LaTeX has nothing left to rescale.
fig, ax = plt.subplots(figsize=(COL_W, COL_W / 2.26))
layers = np.arange(NUM_LAYERS)
for corpus, color, marker, label in styles:
    minor_only, orig_only, n_pairs = classify_first_notes(corpus)
    n = minor_only + orig_only
    ci = [wilson_ci(s, ni) for s, ni in zip(minor_only, n)]
    mean = np.array([c[0] for c in ci])
    half = np.array([c[1] for c in ci])
    print(f'{corpus}: {n_pairs} pairs, {n.sum()} disambiguating notes')
    ax.fill_between(layers, mean - half, mean + half, color=color, alpha=0.2)
    ax.plot(layers, mean, marker, ls='-', color=color, linewidth=1.2, markersize=3, label=label)

ax.set_xticks(range(NUM_LAYERS))
ax.set_xlabel('Layer')
ax.set_ylabel('IIA')
ax.set_ylim(0, 1)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(pad=0.3)
save_fig(fig, 'figure3_iia_by_layer')
plt.show()

---
## Step 7 — Prepare the probing dataset

Builds the inputs for the linear-probing experiments (paper Figure 4). For every chorale in
the **train** and **test** splits — the probe is fit on train and evaluated on test, so the
valid split is not used — this writes:

- `chorale_XXXX/chorale_XXXX_bars_1-N.mid` — the chorale truncated at the end of bar *N*,
  one file per bar. Probe activations are later read at the **last token** of each of these.
- `keys.csv` — one `chorale,key` row per chorale (e.g. `chorale_0000,B- major`). The key
  string supplies the labels for the mode, key and tonic probes; that is all the probes need.

**Bars.** `ticks_per_beat` is 480, but the gcd of all note event times is 120 ticks, so the
real grid is the **16th note** and a 4/4 bar is 16 of them. We cut at every boundary `k*16`
while at least one full bar remains, so a trailing partial bar is never emitted. No chorale
is skipped.


Output: `reproduce_output/probes_data/{train,test}/`. CPU-only — **~1 minute** for both splits.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/prepare_probe_data.py'], check=True)

---
## Step 8 — Extract probe activations

For every bar-truncated MIDI from Step 7, run one forward pass through Aria and save the
**residual stream at the last token**, for all 16 layers. These are the inputs the linear
probes are trained on (paper Figure 4): one activation vector per (chorale, bar, layer).

Output, one file per chorale dir:
`reproduce_output/probes_data/<split>/chorale_XXXX/activations.pt`
— `{'activations': float32 [n_bars, 16, 1536], 'bar_indices': [1, 2, …]}`.

> ⚠️ **Run this step on a GPU, not in the notebook.** From the repo root, in a GPU
> allocation with the aria env active:
>
> ```bash
> reproduce/extract_probe_activations.sh              # both splits
> # reproduce/extract_probe_activations.sh train      # one split only
> ```
>
> **The plain command adapts to the GPUs it sees:**
> - **2+ GPUs** → auto-splits: **train → GPU 0, test → GPU 1** in parallel (each logs to
>   `reproduce_output/probes_data/step8_{train,test}.log`; `tail -f` to watch).
> - **1 GPU** → runs the two splits **sequentially** on that GPU.
> - `SERIAL=1 reproduce/extract_probe_activations.sh` forces one-GPU sequential even with 2 GPUs.
>
> **Time (A10):** ~0.03 s per bar → train (3176 bars) **~1.5 min**, test (1088) **~0.5 min**.
> One GPU **~2 min**; split across 2 GPUs **~1.5 min** (train dominates), plus model load.
> Resumable — re-running skips chorales that already have `activations.pt`
> (pass `--overwrite` to recompute).

---
## Step 9 — Train the linear probes (paper Figures 4 & 6)

Four probe tasks, each a **separately trained** linear classifier per layer (16 layers),
fit on the train-split bar activations and evaluated on test. All labels derive from the
key string in `keys.csv`:

| task | label | classes |
|---|---|---|
| `mode` | `major` / `minor` | 2 |
| `key` | full string, e.g. `B- major` | ~18 |
| `relative` | minor → its **relative** major (`A minor → C major`) — shared key signature | ~9 |
| `parallel` | minor → its **parallel** major (`C minor → C major`) — shared tonic | ~9 |

The grouped probes are retrained on the collapsed labels, not re-evaluations of the full key
probe. Test samples whose label never occurs in train are
dropped (counts printed). Class counts are whatever the data yields — printed, not assumed.

Output: `reproduce_output/probes/probe_results.json` (per-layer test accuracies) and
`probe_weights.pt` (kept for the steering experiments). CPU-only — 64 logistic regressions
on ≤3.2k × 1536 features.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/train_probes.py'], check=True)

In [ ]:
# --- Figure 4: linear probe accuracy across layers (mode vs key) ---
import json
import numpy as np
import matplotlib.pyplot as plt

res = json.load(open('reproduce_output/probes/probe_results.json'))
layers = np.arange(16)
cmap = plt.cm.viridis

# Same fixed \columnwidth x 1.43 in box as Figure 3 -- author at its 2.26 aspect so
# LaTeX has no height left to squash. See the Figure 3 cell for the measurement.
fig, ax = plt.subplots(figsize=(COL_W, COL_W / 2.26))
for task, color, marker, label in [('mode', cmap(0.15), 'o', 'Mode probe'),
                                   ('key', cmap(0.62), 's', 'Key probe')]:
    acc = 100 * np.array(res[task]['accuracy'])
    n_cls = len(res[task]['classes'])
    ax.plot(layers, acc, marker, ls='-', color=color, linewidth=1.2, markersize=3,
            label=f'{label} ({n_cls} classes)')
    pk = int(acc.argmax())
    ax.annotate(f'{acc[pk]:.1f}%', (pk, acc[pk]), textcoords='offset points',
                xytext=(0, 3.5), ha='center', fontsize=BASE_FS - 1, color=color)

ax.set_xlabel('Layer')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(range(16))
ax.set_ylim(0, 100)
ax.legend(loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout(pad=0.3)
save_fig(fig, 'figure4_probe_accuracy')
plt.show()

In [ ]:
# --- Figure 6: relative- vs parallel-grouped key probing ---
import json
import numpy as np
import matplotlib.pyplot as plt

res = json.load(open('reproduce_output/probes/probe_results.json'))
layers = np.arange(16)
cmap = plt.cm.viridis

fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.62))
for task, color, marker, label in [('key', cmap(0.1), 'o', 'Key probe'),
                                   ('relative', cmap(0.5), '^', 'Relative-grouped'),
                                   ('parallel', cmap(0.82), 'v', 'Parallel-grouped')]:
    acc = 100 * np.array(res[task]['accuracy'])
    n_cls = len(res[task]['classes'])
    ax.plot(layers, acc, marker, ls='-', color=color, linewidth=1.2, markersize=3,
            label=f'{label} ({n_cls} classes)')
    pk = int(acc.argmax())
    ax.annotate(f'{acc[pk]:.1f}%', (pk, acc[pk]), textcoords='offset points',
                xytext=(0, 3.5), ha='center', fontsize=BASE_FS - 1, color=color)

ax.set_xlabel('Layer')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(range(16))
ax.set_ylim(0, 100)
ax.legend(loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout(pad=0.3)
save_fig(fig, 'figure6_relative_vs_parallel')
plt.show()

---
## Step 10 — Prepare the steering dataset

The steering experiments (paper Section 5.3, Figure 5) act on **major** chorales, where a
V→I cadence is the natural case for a harmonic modification. This produces the prompts to
steer from: the V→I cadence cuts of every major chorale in the **test** and **eval** splits
(each truncated at the end of the V chord).

It reuses the Step 2 cadence pipeline (`analyze` + `cut`, mode major) but on the test/valid
splits, and **without** minorization — steering adds probe-derived direction vectors to the
residual stream, it does not use the minorized counterfactuals the patching experiment
needed. Non-major chorales are skipped.

Output (new dir): `reproduce_output/steering_data/{test,eval}/`, same layout as the patching
corpus — `chorale_XXXX_chords.txt` + `chorale_XXXX_cuts_output/chorale_XXXX_cut_tick_<N>.mid`.
CPU-only — seconds (test: 42 major chorales → 67 cuts; eval: 38 → 69 cuts).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/prepare_steering_data.py'], check=True)

---
## Step 11 — Build the steering directions

Constructs the steering vectors for the three conditions (paper Section 5.3, Eq. 4) from the
trained probe weights (Step 9). For a source major key and its target, the raw direction at
layer *ℓ* is the difference of two probe-weight rows, and Eq. 4 scales the **unit** direction
to the layer's typical hidden-state magnitude `‖h̄_ℓ‖`:

$$g_\ell = \lVert \bar h_\ell\rVert\,\frac{w_{c_1}-w_{c_2}}{\lVert w_{c_1}-w_{c_2}\rVert}$$

so at generation time the applied vector is just `α · g_ℓ`.

| condition | direction | count |
|---|---|---|
| `mode` | `W_mode[minor] − W_mode[major]` (toward minor) | 1 |
| `relative` | `W_key[relative_minor(key)] − W_key[key]` | 9 |
| `parallel` | `W_key[parallel_minor(key)] − W_key[key]` | 6 |

The magnitude `‖h̄_ℓ‖` is computed from the **training** split (the Step-8
`probes_data/train` activations). Directions are built for every source key in the mapping
tables, independent of which chorales are steered.

Output: `reproduce_output/steering/directions.pt`
(`{h_norm_mean_per_layer, mode, relative, parallel}`). CPU-only — seconds.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/build_steering_directions.py'], check=True)

---
## Step 12 — Steered generation, all conditions (paper Figure 5)

Generate a continuation for every Step-10 V→I cut under all four conditions, saving one
`output.mid` (plus the raw `tokens.json`) per cut. Each condition adds its Step-11 direction
to the residual stream while sampling (`baseline` adds nothing — the reference the others are
compared against):

| condition | direction | α | layers | positions |
|---|---|---|---|---|
| `baseline` | none (unsteered reference) | — | — | — |
| `mode` | toward minor (mode probe) | 0.25 | 5–10 | last bar |
| `relative` | toward relative minor (per key) | 0.15 | 11–15 | all + generated |
| `parallel` | toward parallel minor (per key) | 0.10 | 11–15 | all + generated |


⚠️ **Run on a GPU, not in the notebook.** From the repo root, aria env active:

```bash
reproduce/run_steering.sh
```

Runs all four conditions in one go. Each condition uses both GPUs (test → GPU 0, eval → GPU 1),
running one after another; `SERIAL=1` forces one-GPU sequential. A single condition or subset
can be run explicitly, e.g. `reproduce/run_steering.sh mode`.

> **Time (A10):** ~5 minuts

**Output:** `reproduce_output/steering/<condition>/<split>/<chorale>/<cut>/{output.mid, tokens.json}`.
Resumable — cuts that already have both files are skipped. (`parallel` covers only the 6 major
keys with a parallel-minor probe class; A♭/B♭/E♭-major cuts are skipped.)

---
## Step 13 — Resolution distribution (paper Figure 5)

Label the **first-chord resolution** of every generated continuation (baseline + the three
steered conditions) with a roman numeral relative to the chorale's original **major** key —
the same "resulting harmony" logic as Figures 2/3 — then group the labels into the paper's
six categories and compare distributions.

Categories: **I** (major tonic), **i** (parallel-minor tonic), **vi** (relative-minor tonic),
**orig. & rel. diatonic** (other chords of the shared major/relative-minor collection),
**par. diatonic** (chords of the parallel minor), **other**.

The first cell labels every `output.mid` → `reproduce_output/steering/resolutions.csv`
(CPU, a couple of minutes); the second plots Figure 5. **Requires Step 12 to have run.**

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/label_resolutions.py'], check=True)

In [ ]:
# --- Figure 5: resolution distribution, baseline vs steered ---
import csv, re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# roman-numeral normalisation + the paper's six categories
_PFX = re.compile(r'^([b#♯♭]*[IViv]+)([°oø+]?)(.*)$')
_MG = {'I+': 'I+/bVI+', 'I+7': 'I+/bVI+', 'bVI+': 'I+/bVI+'}
_RO = {'I','ii','iii','IV','V','vi','vii°','vii','I7','ii7','iii7','IV7','V7','vi7',
       'viiø7','vii°7','i','ii°','III','iv','v','VI','VII','i7','III7','IIIø7','iv7',
       'VI7','VII7','bII','bIII','bV','bVI','bVII','bvi','I+/bVI+','bII+','i°','vi°',
       'III°','#i°','#i°7','#ii°','#iv','II','other','none'}
_ST_DIATONIC = {'ii','iii','IV','V','vii°','I7','ii7','iii7','IV7','V7','vi7','viiø7'}
_ST_PAR = {'ii°','iv','v','bIII','bVI','bVII','iv7','i7','i+','v+'}

def _nrm(r):
    if not isinstance(r, str) or not r:
        return None
    m = _PFX.match(r)
    if not m:
        return r
    a, q, rest = m.groups()
    if q == 'o':
        q = '°'
    b = a + q
    s = bool(re.match(r'^(7|65|43|42)(?!\d)', rest))
    return (b + '7') if q == 'ø' else (b + ('7' if s else ''))

def _cn(r):
    if not r:
        return 'none'
    n = _MG.get(_nrm(r), _nrm(r))
    return n if n in _RO else 'other'

def _stack(counter):
    total = sum(counter.values())
    cats = {'I': counter.get('I', 0), 'i': counter.get('i', 0), 'vi': counter.get('vi', 0),
            'orig_rel_diaton': sum(counter.get(r, 0) for r in _ST_DIATONIC),
            'par_diatonic': sum(counter.get(r, 0) for r in _ST_PAR)}
    cats['other'] = max(0, total - sum(cats.values()))
    return cats, total

rows = list(csv.DictReader(open('reproduce_output/steering/resolutions.csv')))
IKEYS = ['I', 'i', 'vi', 'orig_rel_diaton', 'par_diatonic', 'other']
LABELS = ['I', 'i', 'vi', 'org. & rel.\ndiatonic', 'par.\ndiatonic', 'other']
CONDS = [('baseline', 'original'), ('mode', 'mode steering'),
         ('relative', 'relative minor steering'), ('parallel', 'parallel minor steering')]
colors = [plt.cm.viridis(v) for v in (0.1, 0.4, 0.65, 0.85)]

# Four series x six categories with a count over every bar is too dense for one
# column, so this one is authored at \textwidth -- put it in a figure* spanning both
# columns. Set FIG_W = COL_W instead if you'd rather squeeze it into a single column.
FIG_W = TEXT_W
# Text sizes measured off the paper's own Figure 5 by rendering both at the same width
# (465 px) and comparing identical strings. It carries much larger labels than the
# single-column figures, and its aspect ratio is ~2.05, not 2.4.
FS_XTICK, FS_YLABEL, FS_YTICK, FS_LEGEND, FS_COUNT = 14, 15, 11, 15, 9
x, w = np.arange(len(IKEYS)), 0.19
fig, ax = plt.subplots(figsize=(FIG_W, FIG_W / 2.05))
for k, (cond, label) in enumerate(CONDS):
    s, total = _stack(Counter(_cn(r['rn']) for r in rows if r['condition'] == cond))
    pcts = [100 * s[key] / total if total else 0 for key in IKEYS]
    print(f'{label}: n={total}')
    bars = ax.bar(x + (k - 1.5) * w, pcts, w, label=label, color=colors[k])
    for bar, key in zip(bars, IKEYS):
        if s[key] > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(s[key]), ha='center', va='bottom', fontsize=FS_COUNT)
ax.set_xticks(x); ax.set_xticklabels(LABELS, fontsize=FS_XTICK)
ax.tick_params(axis='y', labelsize=FS_YTICK)
ax.set_ylabel('% of samples', fontsize=FS_YLABEL)
ax.set_ylim(0, 100)
ax.legend(loc='upper right', fontsize=FS_LEGEND)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)   # the paper draws light y-gridlines over the bars
fig.tight_layout(pad=0.3)
save_fig(fig, 'figure5_resolution_distribution')
plt.show()

---
## Step 14 — Build the FMD evaluation data (Fréchet Music Distance)

Generation quality is evaluated with **Fréchet Music Distance**
(Retkowski et al. 2024, arXiv:2412.07948), a distributional metric for symbolic music. For
every V→I cut we keep **only the bar after the cut** — 16 sixteenths of continuation, the
material each set actually produced — and write one short MIDI per set, all shifted to start
at t=0. Nothing from before the cut is included, so no real-Bach context is shared between
the reference and the generated sets. This is the FMD variant reported in the paper's Table 1.

Five aligned sets under `reproduce_output/fmd_cont_data/` (files named
`<split>_<chorale>_tick<T>.mid` so the sets line up 1:1):

| set | continuation source |
|---|---|
| `bach` | Bach's real notes after the cut (the reference) |
| `aria` | Aria's **unsteered** continuation |
| `mode` / `relative` / `parallel` | the steered continuations |

FMD is then scored set-vs-set against `bach` (lower = closer to real Bach). CPU-only — **~1 min**
(test: 67 cuts, eval: 69 → 136 pieces per set, 118 for `parallel`). **Requires Step 12 to have run.**

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/build_fmd_cont_data.py'], check=True)

---
## Step 15 — Fréchet Music Distance (FMD) evaluation

Score each generated set from Step 14 against the **bach** reference with **Fréchet Music
Distance** (Retkowski et al. 2024, arXiv:2412.07948), using the CLaMP 2 feature extractor —
lower = closer to real Bach:

| comparison | measures |
|---|---|
| `FMD(bach, aria)` | quality of the **unsteered** continuations |
| `FMD(bach, mode / relative / parallel)` | quality **under** each steering condition |

⚠️ **Run on a GPU, not in the notebook** — it needs the library and the CLaMP 2 model. From
the repo root, aria env active:

```bash
pip install frechet-music-distance     # once; CLaMP 2 (~GB) downloads on the first run
python reproduce/compute_fmd.py        # add --inf for the sample-size-corrected FMD-Inf
```

**Output:** `reproduce_output/fmd_cont_data/fmd_cont_scores.json` + a printed table.

*Reading it:* with ~136 pieces per set the absolute FMD is upward-biased, but every condition
shares the same *n* and the same reference, so the **relative** ordering (aria vs steered) is
the meaningful comparison. **Requires Step 14 to have run.**

---
## Step 16 — Generation-quality table (paper Table 1)

Compute the paper's **Key Steering Evaluation** metrics for baseline / relative / parallel,
with **FMD** as the Quality column. Per-cell definitions match the paper: structural-error rate
(malformed `<pitch,onset,dur>` token triples), degenerate pitch repetition (runs of ≥4 identical
pitches), out-of-range notes (≥3 outside the piano range), average note duration, pitch variance,
and — over the first bar's 16 sixteenth-slots — RN entropy and top-1 dominance.

Reads the full Step-12 generations (`output.mid` + `tokens.json`) and the Step-15
`fmd_cont_scores.json`; writes `reproduce_output/steering/quality_metrics.csv` and prints the table.
CPU-only. **Requires Steps 12 and 15.**

*Note:* the structural-error column needs the raw `tokens.json` — a MIDI round-trip silently
drops malformed tokens — which Step 12 now saves, so struct-error is computed exactly.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/compute_quality_metrics.py'], check=True)

---
## Step 17 — Probe confusion matrices (paper Figure 7)

The key probe's errors are **structured**: it systematically confuses **relative** key pairs
(C major ↔ A minor) but keeps **parallel** keys (C major vs C minor) distinct. Three confusion
matrices, each at its probe's peak layer, make this concrete:

- **Full probe** (18 classes) — ordered so each major sits next to its relative minor, so the
  confusion appears as 2×2 off-diagonal blocks.
- **Relative-grouped** — collapsing relative pairs into one class yields a near-diagonal matrix
  (the confusion is resolved).
- **Parallel-grouped** — collapsing parallel pairs leaves more off-diagonal mass (parallel keys
  were already distinct).

Each is shown as **Recall** (row-normalised) and **Precision** (column-normalised). Reads
`probe_weights.pt` + the Step-8 **test** activations. CPU-only. **Requires Steps 8–9.**

In [ ]:
# --- Figure 7: key-probe confusion matrices (full / relative-grouped / parallel-grouped) ---
import os, sys, json
from collections import Counter
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('reproduce'))
from train_probes import load_split, derive_label

probes = torch.load('reproduce_output/probes/probe_weights.pt', map_location='cpu', weights_only=False)
res = json.load(open('reproduce_output/probes/probe_results.json'))
X_te, keys_te = load_split('reproduce_output/probes_data', 'test')
MIN_COUNT = 5   # keys with fewer test samples fold into "other" (as in the paper's Fig 7)

# Full-probe display order: each major beside its RELATIVE minor, so the model's
# relative-key confusion shows up as 2x2 off-diagonal blocks.
REL_ORDER = ['C major', 'A minor', 'G major', 'E minor', 'D major', 'B minor',
             'A major', 'F# minor', 'E major', 'C# minor', 'F major', 'D minor',
             'B- major', 'G minor', 'E- major', 'C minor', 'A- major', 'F minor']

def confusion(task, order=None):
    """Confusion matrices for `task` at its PEAK-accuracy layer."""
    W, b = probes[task]['W'], probes[task]['b']
    classes = [str(c) for c in probes[task]['classes']]
    peak = int(np.argmax(res[task]['accuracy']))
    y = np.array([derive_label(task, k) for k in keys_te])
    keep = np.isin(y, classes)
    y = y[keep]
    Xl = X_te[torch.from_numpy(keep)][:, peak].float()
    pred = np.array(classes)[(Xl @ W[peak].T + b[peak]).argmax(1).numpy()]
    cnt = Counter(y)
    main = [c for c in (order or classes) if c in classes and cnt.get(c, 0) >= MIN_COUNT]
    folded = [c for c in classes if c not in main]
    disp = main + (['other'] if folded else [])          # only add "other" if something folds
    di = {c: i for i, c in enumerate(disp)}
    bucket = lambda c: c if c in main else 'other'
    cm = np.zeros((len(disp), len(disp)))
    for t, p in zip(y, pred):
        cm[di[bucket(t)], di[bucket(p)]] += 1
    recall = cm / cm.sum(1, keepdims=True).clip(min=1)
    precision = cm / cm.sum(0, keepdims=True).clip(min=1)
    return disp, recall, precision

def short(lbls):
    return [l.replace(' major', ' maj').replace(' minor', ' min') for l in lbls]

panels = [('key', 'Full probe', REL_ORDER, 'full'),
          ('relative', 'Relative-grouped', None, 'relative'),
          ('parallel', 'Parallel-grouped', None, 'parallel')]

# One figure per probe, each keeping its recall/precision pair, so the three can be
# placed and captioned independently. Authored at \textwidth (figure* spanning both
# columns): 19 class labels per axis do not survive a single column.
for task, title, order, slug in panels:
    disp, recall, precision = confusion(task, order)
    fig, axes = plt.subplots(1, 2, figsize=(TEXT_W, TEXT_W * 0.52))
    for ax, (M, kind) in zip(axes, [(recall, 'Recall'), (precision, 'Precision')]):
        im = ax.imshow(M, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(len(disp))); ax.set_xticklabels(short(disp), rotation=90, fontsize=BASE_FS - 1)
        ax.set_yticks(range(len(disp))); ax.set_yticklabels(short(disp), fontsize=BASE_FS - 1)
        ax.set_title(f'{title} — {kind}')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(pad=0.3)
    save_fig(fig, f'figure7_probe_confusion_{slug}')
    plt.show()

---
## Step 18 — Single-layer steering: is one layer enough? (paper Sec. 5.4)

Re-runs the `parallel` and `relative` steering conditions with the direction added at **one
layer only (layer 12)**, at α = 0.4 — larger than the full-range α on purpose, since 0.10
added at each of five layers is a much bigger total intervention than 0.10 added at one.
This is the experiment behind the paper's "a single layer carries most of the parallel-minor
effect, but relative minor needs the whole range" claim.

⚠️ **Run on a GPU, not in the notebook.** From the repo root, aria env active:

```bash
reproduce/run_single_layer_steering.sh     # layer 12, alpha 0.40
```

The script generates into `reproduce_output/steering_l12/`, labels the resolutions, builds
the `parallel_l12` / `relative_l12` FMD sets beside the main ones (same `bach/` reference),
merges their scores into `fmd_cont_scores.json`, and computes the Table-1 metrics for them.
**Requires Steps 11–12 and 14–15 to have run.**

The cell below (CPU-only) then prints the per-category resolution counts quoted in the paper
text — e.g. **76 / 118** cuts resolving to the parallel-minor tonic under layer-12-only
steering — and writes the layer-12-vs-full-range comparison figure.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'reproduce/plot_single_layer.py'], check=True)